In [1]:
import pandas as pd
from krippendorff_alpha.compute_alpha import compute_alpha
from krippendorff_alpha.preprocessing import ColumnMapping

In [2]:
# ---------------------------------------------------------
# 1. Load the Excel Workbook
# ---------------------------------------------------------
excel_path = "/home/david/Git/SDLT_Paper/Data/adiya_isla_scores.xlsx"
xls = pd.ExcelFile(excel_path)

# List of sheets to evaluate (skipping overview sheets like 'Whole' if present)
sheets_to_process = [
    "CoC", "Au", "In", "Re", "La", "Fi", "Cq", "Rr", "Pc"
]

# ---------------------------------------------------------
# 2. Configuration Settings
# ---------------------------------------------------------
annotator_cols = ["annotator1", "annotator2", "annotator3"]
scale = ["0", "1", "2"]

column_mapping = ColumnMapping(
    text_col="Publication",
    annotator_cols=annotator_cols
)

In [3]:
# ---------------------------------------------------------
# 3. Compute Krippendorff's Alpha per Sheet and Round
# ---------------------------------------------------------
results_list = []

for sheet_name in sheets_to_process:
    if sheet_name not in xls.sheet_names:
        print(f"Warning: Sheet '{sheet_name}' not found in workbook. Skipping.")
        continue

    # Read sheet with two-level headers (Row 1: Round, Row 2: Annotator)
    df_raw = pd.read_excel(xls, sheet_name=sheet_name, header=[0, 1])

    # Extract the Publication column values
    pub_col_name = [col for col in df_raw.columns if "Publication" in str(col)][0]
    publication_series = df_raw[pub_col_name]

    row_result = {"Category": sheet_name}

    # Iterate over both rounds
    for round_name in ["Round 1", "Round 2"]:
        # Extract the columns belonging to the current round
        df_round = df_raw[round_name].copy()
        df_round["Publication"] = publication_series.values

        # Convert rating values to string format for the library
        df_round[annotator_cols] = df_round[annotator_cols].astype(str)

        # Compute Krippendorff's alpha for ordinal data
        res = compute_alpha(
            df=df_round,
            data_type="ordinal",
            column_mapping=column_mapping,
            ordinal_scale=scale
        )

        # Append round result to row dictionary
        col_key = f"Alpha_{round_name.replace(' ', '_')}"
        row_result[col_key] = res["alpha"]

    results_list.append(row_result)

In [4]:
# ---------------------------------------------------------
# 4. Format & Display Results
# ---------------------------------------------------------
df_results = pd.DataFrame(results_list)

print("\n--- Krippendorff's Alpha Results ---")
print(df_results.to_string(index=False))


--- Krippendorff's Alpha Results ---
Category  Alpha_Round_1  Alpha_Round_2
     CoC          0.899          1.000
      Au          0.851          1.000
      In          0.951          1.000
      Re          0.636          1.000
      La          0.724          1.000
      Fi          0.890          1.000
      Cq          0.713          1.000
      Rr          0.906          1.000
      Pc          0.729          0.978
